In [1]:
import numpy as np
import pyvista as pv
import hyperzeta.plot as hzplt
import hyperzeta as hz
import mlx.core as mx
from tqdm.auto import trange

pv.set_jupyter_backend("client")

In [2]:
beta = 0.99
j = 2.0

sphere = hzplt.make_icosphere(subdivisions=6, radius=1.0)
cj = hz.QedCoef()
cj.log = True
p = cj.tune(j, beta * np.array([1.0, 0.0, 0.0]), residual=1.0e-2, device=mx.gpu)
cj.log = False
n_pts = sphere.points.shape[0]
values = np.empty(n_pts)
for i in trange(n_pts):
    dir = np.array(sphere.points[i])
    values[i] = cj(j, beta * dir, p, device=mx.gpu)


[QedCoef] eta= 1.0000 nmax= 15 c_j=7.688772446685417e+01
[QedCoef] eta= 0.9091 nmax= 15 c_j=7.742514469607188e+01 residual= 6.62e-02
[QedCoef] eta= 0.8333 nmax= 15 c_j=7.777770552081117e+01 residual= 4.32e-02
[QedCoef] eta= 0.7692 nmax= 15 c_j=7.800661916995239e+01 residual= 2.79e-02
[QedCoef] eta= 0.7143 nmax= 15 c_j=7.815497461778963e+01 residual= 1.81e-02
[QedCoef] eta= 0.6667 nmax= 15 c_j=7.825168083104785e+01 residual= 1.18e-02
[QedCoef] eta= 0.6250 nmax= 15 c_j=7.831551987135418e+01 residual= 7.75e-03


  0%|          | 0/40962 [00:00<?, ?it/s]

In [3]:
name = f"c_{j:.0f}"
sphere[name] = values
cmin = values.min()
cmax = values.max()
relief = sphere.compute_normals(cell_normals=False, point_normals=True)
relief = relief.warp_by_scalar(name, factor=0.5 / cmax)


In [4]:
pl1 = pv.Plotter(notebook=True, window_size=[600, 600])
pl1.add_mesh(
    sphere,  # type: ignore
    scalars=name,
    cmap=hzplt.make_rwb_cmap(cmin, cmax),
    clim=[cmin, cmax],
    smooth_shading=True,
    show_edges=False,
)
pl1.show()

Widget(value='<iframe src="http://localhost:57939/index.html?ui=P_0x135e5c1a0_0&reconnect=auto" class="pyvista…

In [5]:
pl2 = pv.Plotter(notebook=True, window_size=[600, 600])
pl2.add_mesh(
    relief,  # type: ignore
    scalars=name,
    cmap=hzplt.make_rwb_cmap(cmin, cmax),
    clim=[cmin, cmax],
    smooth_shading=True,
    show_edges=False,
)
pl2.show()

Widget(value='<iframe src="http://localhost:57939/index.html?ui=P_0x137e7a5d0_1&reconnect=auto" class="pyvista…